# Real street networks: a bounded routing example

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/networks/real-life-shortest-path.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/networks/real-life-shortest-path.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Optional live-data extension
This notebook needs internet access and queries OpenStreetMap through OSMnx. It is **not part of the offline execution guarantee**. Live map data and service availability change; for a fixed assignment, the instructor should retain an approved GraphML snapshot with its retrieval date and OpenStreetMap attribution.

The earlier AABW and Heuristics notebooks duplicated geocoding, city-wide downloads and large many-to-many experiments. This shared entry point uses a small area, fixed coordinates and the current public OSMnx API. It makes no hidden requests to geocoding services. See [OSMnx documentation](https://osmnx.readthedocs.io/en/stable/user-reference.html) and [OpenStreetMap attribution and licence](https://www.openstreetmap.org/copyright).


## Setup

Colab already provides the general-purpose scientific libraries. This cell ensures the tested OSMnx version for the optional map extension; pip keeps an already-installed matching version. It does not replace Colab's NumPy, pandas or Matplotlib just to match the maintenance environment.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
from importlib.util import find_spec
import subprocess
import sys

required_packages = {
    'networkx': 'networkx',
    'osmnx': 'osmnx',
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])


In [ ]:
from pathlib import Path
import osmnx as ox
import networkx as nx
ox.settings.use_cache = True
ox.settings.requests_timeout = 60
snapshot = Path('amsterdam-teaching-walk.graphml')
if snapshot.is_file():
    graph = ox.load_graphml(snapshot)
else:
    # Small area around the Roeterseiland area; never download a whole country here.
    graph = ox.graph_from_point((52.3635,4.9115),dist=600,network_type='walk')
    ox.save_graphml(graph,snapshot)
print(graph.number_of_nodes(), graph.number_of_edges())


In [ ]:
source = ox.distance.nearest_nodes(graph,4.9095,52.3645)
target = ox.distance.nearest_nodes(graph,4.9140,52.3620)
route = nx.shortest_path(graph,source,target,weight='length')
route_edges = ox.routing.route_to_gdf(graph,route,weight='length')
print('Network length in metres:', route_edges['length'].sum())
ox.plot_graph_route(graph,route,node_size=0)


## Questions to ask before trusting a route
How far were the coordinates snapped to the network? Is the walking graph connected? Are crossings and access restrictions represented as expected? Does the shortest network route correspond to the real journey you intend to model?

For a many-to-many extension, first collapse parallel arcs consistently and preserve direction before comparing another engine such as Pandana. Compare distances, not equality of node sets: two equally short routes can differ. Large routing experiments belong in the course or research project that needs them, not in every beginner's setup.
